In [8]:
!pip install transformers tensorflow tf-keras safetensors

# 1. TFTLite란?
- TFLite (TensorFlow Lite): TensorFlow 모델을 자원이 제한된 환경(모바일, IoT, 웹 브라우저)에서 실행할 수 있도록 변환한 '경량화 포맷'

## 1-1. 왜 필요한가?
1. 크기 (Size): TFLite로 변환(특히 양자화) 진행 시 파일 크기가 1/4 이하로 축소. (예: 260MB → 65MB)
2. 속도 (Speed): 모바일 기기의 NPU나 GPU 가속을 받도록 최적화되어, 서버 없이도 기기 자체에서 빠르게 추론할 수 있음
3. 호환성 (Compatibility): Python이 설치되지 않은 환경(JavaScript, Android, iOS)에서 AI 모델을 실행할 수 있게 해주는 '엔진' 역할

# 2. TFLite를 활용한 모델 경량화
## 2-1. `distilbert-base-uncased-distilled-squad` 모델
1. 경량화된 DistilBERT 모델로, SQuAD 데이터셋에 대해 증류 학습됨
2. 용도: 질문과 답변
3. DistilBERT: BERT 경량화 버젼
4. SQuAD: Stanford Question Answering Dataset
5. 증류 학습: 큰 모델의 지식을 작은 모델에 전달하는 기법


## 2-2. 작업 순서
1. 모델/토크나이저 로드

In [2]:
from google.colab import drive

# 1. Google Drive 마운트
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/Colab Notebooks/AI/14_TFLite/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
from transformers import TFAutoModelForQuestionAnswering, AutoTokenizer


# 기본 설정
MODEL_NAME = "distilbert-base-uncased-distilled-squad"
SAVEDMODEL_DIR = base_path + "/saved_model_distilbert"
TFLITE_PATH = base_path + "/qa_model.tflite"
TOKENIZER_DIR = base_path + "/distilbert_tokenizer"
MAX_LEN = 384

os.makedirs(os.path.dirname(TFLITE_PATH) or ".", exist_ok=True)

print("모델/토크나이저 로드")
model = TFAutoModelForQuestionAnswering.from_pretrained(
        MODEL_NAME,
        from_pt=True,  # PyTorch 가중치 로드를 명시
        use_safetensors=False # safetensors 사용 비활성화
    )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

모델/토크나이저 로드


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFDistilBertForQuestionAnswering.

All the weights of TFDistilBertForQuestionAnswering were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForQuestionAnswering for predictions without further training.


2. 그래프 트레이싱 (더미 데이터로 모델 생성)
    - SavedModel로 내보낼 때 그래프가 필요하기 때문에 진행
    - TFLite 같은 배포 환경은 모든 연산 순서가 미리 고정된 **정적 그래프(Static Graph)**가 필요
    - 그래프란: 모델의 연산 흐름을 정의한 구조

In [7]:
def build_and_trace(model, tokenizer):
    # 더미 입력으로 그래프 빌드 (Tracing)
    inputs = tokenizer(
        "hello",
        "world",
        return_tensors="tf",        # TensorFlow 텐서 반환
        padding="max_length",       # 최대 길이로 패딩
        truncation="only_second",   # 컨텍스트가 길 때만 자르도록 지정
        max_length=MAX_LEN,
    )
    _ = model(inputs)

build_and_trace(model, tokenizer)

3. SavedModel 내보내기
    - TFLite로 변환 과정에서 원본 모델을 참조
    - 서빙 시그니처 정의
        - 모델을 서빙(배포)할 때 사용할 입력/출력 형식을 정의한 것
        - TensorFlow Serving이나 TFLite에서 모델을 호출할 때 이 시그니처를 참고하여 올바른 형식의 데이터를 전달하고 결과를 받을 수 있도록 함

In [8]:
import tensorflow as tf


def export_saved_model(model):
    # 서빙 시그니처 정의
    @tf.function(input_signature=[{
        "input_ids": tf.TensorSpec(shape=[None, None], dtype=tf.int32),
        "attention_mask": tf.TensorSpec(shape=[None, None], dtype=tf.int32),
    }])
    def serving_fn(features):
        outputs = model(features)
        return {
            "start_logits": outputs.start_logits,
            "end_logits": outputs.end_logits,
        }

    tf.saved_model.save(model, SAVEDMODEL_DIR, signatures={"serving_default": serving_fn})

export_saved_model(model)

4. TFLite 변환
    - TFLiteConverter: TensorFlow 모델을 TFLite 형식으로 변환하는 도구
    - 웹/모바일 호환성: 일부 연산은 기본 TFLite에 없으므로 SELECT_TF_OPS 포함
        - SELECT_TF_OPS: TensorFlow의 일부 연산을 TFLite에서 사용할 수 있도록 허용
        - 예: 복잡한 수학 연산, 특정 신경망 연산 등
    - 동적 범위 양자화 (dyamic)
        - 모델의 가중치(파라미터)를 32비트 부동소수점(Float32)에서 **8비트 정수(Int8)**로 '압축'
        - 모델이 실행되는 순간(Dynamic)에 가중치의 범위를 계산해서 정수로 변환하기 때문에 '동적'

### [참고] TFLiteConverter를 사용하지 않는다면...
- `converter.convert()`를 호출하는 순간, '동적 양자화(숫자 압축)'뿐만 아니라 '구조적 최적화'도 함께 수행

- 구조적 최적화를 직접 진행한다면, 'Fused QKV'를 사용
1. 원본 모델: 어텐션을 계산하기 위해 Query, Key, Value (QKV) 레이어를 3개의 분리된 레이어로 각각 연산
2. 문제점:
    1. 모바일 칩에서는 이 3개의 연산을 위해 메모리를 3번 접근해야 함
    2. 이 메모리 접근(I/O)이 연산 속도보다 느린 '병목 현상'을 발생시킴
3. 최적화: TFLiteConverter는 이 3개의 레이어를 물리적으로 '1개의 통합된 레이어'로 자동 병합(Fusion)
4. 결과: 메모리 접근이 1번으로 줄어들어, 전력 소모가 감소하고 추론 속도가 향상

(진행 방법이 궁금하다면, 기존 실습 참고)

In [9]:
def convert_to_tflite(select_tf_ops=True, quantize=None):
    # TFLite 변환기 생성
    converter = tf.lite.TFLiteConverter.from_saved_model(SAVEDMODEL_DIR)

    # 연산 호환성 설정
    if select_tf_ops:
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]

    # 선택적 양자화
    if quantize == "dynamic":       # 동적 범위 양자화
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif quantize == "float16":     # float16 양자화
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    # 정수 양자화는 대표 데이터셋 함수가 필요하므로 본 실습 범위에선 제외

    tflite_model = converter.convert()
    with open(TFLITE_PATH, "wb") as f:
        f.write(tflite_model)
    print(f"TFLite 모델 저장: {TFLITE_PATH}")

convert_to_tflite(quantize='dynamic')

TFLite 모델 저장: /content/drive/MyDrive/Colab Notebooks/AI/14_TFLite//qa_model.tflite


5. 토크나이저와 샘플 내보내기
    - 경량화 모델을 활용한 서비스에서 사용할 수 있도록, 토크나이저와 샘플을 별도로 저장
    - ex) 모바일, 웹 등

In [10]:
import json

# 토크나이저 내보내기
os.makedirs(TOKENIZER_DIR, exist_ok=True)
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"토크나이저 저장: {TOKENIZER_DIR}")

# 샘플 입력 저장
sample = {
    "question": "Who wrote the book?",
    "context": "The book was written by Alan Turing in 1950.",
}
os.makedirs("../sample", exist_ok=True)
with open("../sample/qa_examples.json", "w") as f:
    json.dump([sample], f, indent=2)
print("샘플 저장: ../sample/qa_examples.json")

토크나이저 저장: /content/drive/MyDrive/Colab Notebooks/AI/14_TFLite//distilbert_tokenizer
샘플 저장: ../sample/qa_examples.json


# Extra. node 환경에서 경량화 모델 사용하기 (진행하지 않음)
## E-1. 왜 진행하지 않나요?
1. 복잡한 라이브러리 설치
    - `@tensorflow/tfjs-node`, `@tensorflow/tfjs-tflite`, `@xenova/transformers`
    - colab 환경에서 활용했던 TFLite와 transfomers를 node 환경에서도 동일하게 설치하여 사용하여야 함.
    - 이때, 단순히 라이브러리를 설치하는 것 뿐만아니라, 현재 OS환경에 맞는 빌드 과정이 필요
    - 교육장 환경 및 개인 PC 설정등에 따라 빌드가 불가능한 경우가 많음.
2. Wasm / 네이티브 C++ 종속성 문제
    - 위의 라이브러리 등은 고성능을 위하여 WebAssembly를 사용
    - WebAssembly란, C, C++, Rust 같은 고성능 언어로 작성된 코드를 웹 브라우저에서 직접 실행할 수 있도록 컴파일한 바이너리(기계어) 파일 형식
    - Wasm은 편리하지만, 버젼 의존성이 매우 높아 테스트 환경에서는 불안정 함.
    - 또한, 강의장 환경 특성상 브라우저 보안 정책에 의해 파일 로드가 되지 않을 가능성이 매우 높음.

## E-2. 코드 둘러보기
- `app.js` 참고